In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import sys

sys.path.append("..")

In [12]:

from constrerl.utils import extract_flags_from_name, task_id_to_name
import glob
from pathlib import Path
import json
import pandas as pd
from collections.abc import Callable, Awaitable


In [13]:
test_file = Path("./results_test/all.csv")
test_df = pd.read_csv(test_file)
report_dir = Path("report")

# test_df = test_df[(test_df["Team ID"] == "TUGW") |  (test_df["Team ID"] == "Organizers")]
# report_dir = report_dir / "test-tugw"
# lbl_xtra = ":test:tugw"
# mode = "Merged Test"
test_df = test_df[(test_df["Team ID"] == "ToGS") |  (test_df["Team ID"] == "Organizers")]
report_dir = report_dir / "test"
lbl_xtra = ":test:togs"
mode = "Test"

report_dir.mkdir(exist_ok=True, parents=True)
test_df

,Team ID,Task ID,Run ID,System Description,macro_precision,macro_recall,macro_f1,micro_precision,micro_recall,micro_f1
0,Organizers,T611,BASELINE,GLiNER,0.711354,0.747989,0.726653,0.778246,0.822090,0.799567
69,ToGS,T611,hermesbeamnonehermes318Bbase,CHASTE,0.103839,0.029667,0.023255,0.034678,0.018162,0.023838
70,ToGS,T611,hermesbeamnonehermes323Bbase,CHASTE,0.045921,0.019983,0.012508,0.030994,0.012602,0.017918
71,ToGS,T611,hermesloraentitiesnaivebeamnonehermes323Bentities,CHASTE,0.323552,0.391645,0.333386,0.342805,0.554485,0.423676
72,ToGS,T611,hermesnaivebeamnonehermes318Bbase,CHASTE,0.276523,0.415946,0.307339,0.289350,0.480356,0.361154
...,...,...,...,...,...,...,...,...,...,...
409,ToGS,T622,hermesraglorasbeamnonehermes323Bs,CHASTE,0.017780,0.025772,0.010901,0.048892,0.052675,0.050713
410,ToGS,T622,hermesraglorasnaivebeamnonehermes318Bs,CHASTE,0.020579,0.032821,0.014476,0.054217,0.059259,0.056626
411,ToGS,T622,hermesraglorasnaivebeamnonehermes323Bs,CHASTE,0.017780,0.025772,0.010901,0.048892,0.052675,0.050713
412,ToGS,T622,hermesragnaivebeamnonehermes318Bbase,CHASTE,0.051441,0.037719,0.034030,0.036324,0.054321,0.043536


In [14]:
eval_results: list[dict] = []


score_map = {
    "macro_precision": "$P$",
    "macro_recall": "$R$",
    "macro_f1": "$F_1$",
    "micro_precision": "$P_{micro}$",
    "micro_recall": "$R_{micro}$",
    "micro_f1": "$F_{1,micro}$",
}


def test_table_to_df(
    table: pd.DataFrame,
    task: str,
) -> pd.DataFrame:
    eval_results = []
    for i, row in table.iterrows():
        run_id: str = row["Run ID"]
        merge_mode = "Merged" in mode
        if row["Task ID"] != task:
            continue
        eval_result = {f"{k}": v for k, v in row.items() if k in score_map}

        result_dict = extract_flags_from_name(
            run_id, merge_mode=merge_mode, k=None, test_mode=True
        )
        result_dict.update(eval_result)
        eval_results.append(result_dict)
    if len(eval_results) == 0:
        return pd.DataFrame()
    eval_df = pd.DataFrame(eval_results)
    # remove duplicate rows
    eval_df = eval_df.drop_duplicates()
    eval_df.rename(score_map, axis=1, inplace=True)
    valid_cols = [
        c
        for c in [
            "Graphwise",
            "Set",
            "Model",
            "Beams",
            "NED FT",
            "RAG",
            "LoRA",
            "Naive",
            "Filter",
        ]
        if c in eval_df.columns
    ]
    eval_df.set_index(valid_cols, inplace=True)
    eval_df = eval_df.sort_index()
    
    # if "$F_{1,micro}$" in eval_df.columns:
    #     eval_df = eval_df.sort_values("$F_{1,micro}$")
    return eval_df


task_6_1_1_df = test_table_to_df(test_df, "T611")
task_6_1_2_df = test_table_to_df(test_df, "T612")
task_6_2_1_df = test_table_to_df(test_df, "T621")
task_6_2_2_df = test_table_to_df(test_df, "T622")
task_6_1_1_df

$P$  \
Model     Beams    NED FT     RAG        LoRA       Naive      Filter                 
 Baseline -        -          -          -          -          -           0.711354   
3.1 8B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.103839   
                                                    \checkmark $\times$    0.276523   
                              \checkmark $\times$   $\times$   $\times$    0.374668   
                                                    \checkmark $\times$    0.335337   
                                         \checkmark $\times$   $\times$    0.197714   
                   \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                    \checkmark $\times$    0.306500   
                                         \checkmark \checkmark $\times$    0.305095   
                              \checkmark $\times$   $\times$   $\times$    0.003497   
                                         \checkmark \checkmark $\times$    0.327929   
3.2 3B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.045921   
                                                    \checkmark $\times$    0.280636   
                                         \checkmark \checkmark $\times$    0.323552   
                              \checkmark $\times$   $\times$   $\times$    0.322846   
                                                    \checkmark $\times$    0.333952   
                                         \checkmark $\times$   $\times$    0.113513   
                                                    \checkmark $\times$    0.318824   
                   \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                    \checkmark $\times$    0.300719   
                                         \checkmark \checkmark $\times$    0.332628   
                              \checkmark $\times$   $\times$   $\times$    0.004274   
                                         \checkmark \checkmark $\times$    0.326303   
Naive     $\times$ $\times$   $\times$   $\times$   \checkmark $\times$    0.307182   
                                                               \checkmark  0.284171   

                                                                                $R$  \
Model     Beams    NED FT     RAG        LoRA       Naive      Filter                 
 Baseline -        -          -          -          -          -           0.747989   
3.1 8B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.029667   
                                                    \checkmark $\times$    0.415946   
                              \checkmark $\times$   $\times$   $\times$    0.223773   
                                                    \checkmark $\times$    0.450644   
                                         \checkmark $\times$   $\times$    0.070961   
                   \checkmark $\times$   $\times$   $\times$   $\times$    0.000000   
                                                    \checkmark $\times$    0.447209   
                                         \checkmark \checkmark $\times$    0.453021   
                              \checkmark $\times$   $\times$   $\times$    0.001789   
                                         \checkmark \checkmark $\times$    0.448721   
3.2 3B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$    0.019983   
                                                    \checkmark $\times$    0.417106   
                                         \checkmark \checkmark $\times$    0.391645   
                              \checkmark $\times$   $\times$   $\times$    0.174418   
                                                    \checkmark $\times$    0.441260   
                                         \checkmark $\times$   $\times$    0.069524   
                                                    \checkmark $\times$    0.

In [15]:
task_6_2_2_df

$P$  \
Model     Beams    NED FT     RAG        LoRA       Naive      Filter               
 Baseline -        -          -          -          -          -         0.100907   
3.1 8B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.001294   
                                                    \checkmark $\times$  0.001294   
                                         \checkmark $\times$   $\times$  0.045429   
                                                    \checkmark $\times$  0.045429   
                              \checkmark $\times$   $\times$   $\times$  0.051441   
                                                    \checkmark $\times$  0.051441   
                                         \checkmark $\times$   $\times$  0.020579   
                                                    \checkmark $\times$  0.020579   
                   \checkmark $\times$   $\times$   $\times$   $\times$  0.000212   
                                                    \checkmark $\times$  0.000228   
                              \checkmark $\times$   $\times$   $\times$  0.009127   
                                         \checkmark \checkmark $\times$  0.018646   
3.2 3B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                    \checkmark $\times$  0.000000   
                                         \checkmark $\times$   $\times$  0.028328   
                                                    \checkmark $\times$  0.028328   
                              \checkmark $\times$   $\times$   $\times$  0.032726   
                                                    \checkmark $\times$  0.032726   
                                         \checkmark $\times$   $\times$  0.017780   
                                                    \checkmark $\times$  0.017780   
                   \checkmark $\times$   $\times$   $\times$   $\times$  0.000000   
                                                    \checkmark $\times$  0.000000   
                                         \checkmark \checkmark $\times$  0.012071   
                              \checkmark $\times$   $\times$   $\times$  0.020742   
                                         \checkmark \checkmark $\times$  0.000515   

                                                                              $R$  \
Model     Beams    NED FT     RAG        LoRA       Naive      Filter               
 Baseline -        -          -          -          -          -         0.102058   
3.1 8B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.022692   
                                                    \checkmark $\times$  0.022692   
                                         \checkmark $\times$   $\times$  0.034707   
                                                    \checkmark $\times$  0.034707   
                              \checkmark $\times$   $\times$   $\times$  0.037719   
                                                    \checkmark $\times$  0.037719   
                                         \checkmark $\times$   $\times$  0.032821   
                                                    \checkmark $\times$  0.032821   
                   \checkmark $\times$   $\times$   $\times$   $\times$  0.005325   
                                                    \checkmark $\times$  0.007692   
                              \checkmark $\times$   $\times$   $\times$  0.007761   
                                         \checkmark \checkmark $\times$  0.003238   
3.2 3B    $\times$ $\times$   $\times$   $\times$   $\times$   $\times$  0.000000   
                                                    \checkmark $\times$  0.000000   
                                         \checkmark $\times$   $\times$  0.038632   
                                                    \checkmark $\times$  0.038632   
                              \checkmark $\times$   $\times$   $\times$  0.037040 

In [16]:
tasks = {
    "6.1.1": task_6_1_1_df,
    "6.1.2": task_6_1_2_df,
    "6.2.1": task_6_2_1_df,
    "6.2.2": task_6_2_2_df,
}
average_improvements = []

import re

from constrerl.utils import calculate_improvements, df_topk


for task_name, task_df in tasks.items():
    if len(task_df) == 0:
        print(f"No results for Subtask {task_name}, skipping.")
        continue
    print(f"Processing Subtask {task_name} with {len(task_df)} results.")
    task_name_formatted = re.sub(r"\.", "_", task_name)

    task_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(
        lambda v: f"{v:.2f}" if isinstance(v, float) else v, na_rep="-"
    ).to_latex(
        report_dir / f"task_{task_name_formatted}_full.tex",
        caption=f"{mode} Set Results for {task_id_to_name(task_name)} (Subtask {task_name})",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}:full",
        clines="all;data",
        hrules=True,
    )
    top_k = 10
    task_df_top = df_topk(task_df, top_k)
    task_df_top.style.highlight_max(axis=0, props="textbf:--rwrap;").format(
        na_rep="-", precision=2
    ).to_latex(
        report_dir / f"subtask_{task_name_formatted}_top.tex",
        caption=f"Top {top_k} {mode} Set Results for {task_id_to_name(task_name)} (Subtask {task_name})",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}",
        clines="all;data",
        hrules=True,
    )
    task_df_improved = calculate_improvements(task_df)
    mean_improvements = task_df_improved.mean()

    task_df_improved.style.format(
        lambda v: (
            f"+\\textcolor{{DarkGreen}}{{{v:.2f}}}"
            if v > 0
            else f"\\textcolor{{DarkRed}}{{{v:.2f}}}"
            if isinstance(v, float)
            else v
        ),
        na_rep="-",
    ).to_latex(
        report_dir / f"task_{task_name_formatted}_improve.tex",
        caption=f"{mode} Set Improvements for {task_id_to_name(task_name)} (Subtask {task_name})",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}_improve",
        clines="all;data",
        hrules=True,
    )
    average_improvements.append(mean_improvements.to_dict() | {"Subtask": task_name})

Processing Subtask 6.1.1 with 25 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', ['-'])] / other keys: ['Model', 'NED FT', 'RAG', 'LoRA', 'Naive', 'Filter']
No improvements calculated. Returning empty DataFrame.
Processing Subtask 6.1.2 with 25 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', ['-'])] / other keys: ['Model', 'NED FT', 'RAG', 'LoRA', 'Naive', 'Filter']
No improvements calculated. Returning empty DataFrame.
Processing Subtask 6.2.1 with 26 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', ['-'])] / other keys: ['Model', 'NED FT', 'RAG', 'LoRA', 'Naive', 'Filter']
No improvements calculated. Returning empty DataFrame.
Processing Subtask 6.2.2 with 26 results.
Calculating improvements based on keys: ['Beams'] with options: [('Beams', ['-'])] / other keys: ['Model', 'NED FT', 'RAG', 'LoRA', 'Naive', 'Filter']
No improvements calculated. Returning empty DataFrame.


In [17]:
task_df_top

$P$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter               
 Baseline -        -        -          -          -          -         0.100907   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.045429   
                                                  \checkmark $\times$  0.045429   
                            \checkmark \checkmark $\times$   $\times$  0.020579   
                                                  \checkmark $\times$  0.020579   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.028328   
                                                  \checkmark $\times$  0.028328   
                            \checkmark $\times$   $\times$   $\times$  0.032726   
                                       \checkmark $\times$   $\times$  0.017780   
                                                  \checkmark $\times$  0.017780   

                                                                            $R$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter               
 Baseline -        -        -          -          -          -         0.102058   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.034707   
                                                  \checkmark $\times$  0.034707   
                            \checkmark \checkmark $\times$   $\times$  0.032821   
                                                  \checkmark $\times$  0.032821   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.038632   
                                                  \checkmark $\times$  0.038632   
                            \checkmark $\times$   $\times$   $\times$  0.037040   
                                       \checkmark $\times$   $\times$  0.025772   
                                                  \checkmark $\times$  0.025772   

                                                                          $F_1$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter               
 Baseline -        -        -          -          -          -         0.096610   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.025062   
                                                  \checkmark $\times$  0.025062   
                            \checkmark \checkmark $\times$   $\times$  0.014476   
                                                  \checkmark $\times$  0.014476   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$  0.026640   
                                                  \checkmark $\times$  0.026640   
                            \checkmark $\times$   $\times$   $\times$  0.027387   
                                       \checkmark $\times$   $\times$  0.010901   
                                                  \checkmark $\times$  0.010901   

                                                                       $P_{micro}$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter                  
 Baseline -        -        -          -          -          -            0.140304   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$     0.059775   
                                                  \checkmark $\times$     0.059775   
                            \checkmark \checkmark $\times$   $\times$     0.054217   
                                                  \checkmark $\times$     0.054217   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$     0.060232   
                                                  \checkmark $\times$     0.060232   
                            \checkmark $\times$   $\times$   $\times$     0.037923   
                                       \checkmark $\times$   $\times$     0.048892   
                                                  \checkmark $\times$     0.048892   

                                                    

In [18]:
import re

from constrerl.utils import calculate_relative_improvements

pairs = {
    "Subtask 6.1": [task_6_1_1_df, task_6_1_2_df],
    "Subtask 6.2": [task_6_2_1_df, task_6_2_2_df],
}
for task_name, (df1, df2) in pairs.items():
    if df1.empty or df2.empty:
        print(f"Skipping {task_name} due to empty DataFrame.")
        continue
    task_name_formatted = re.sub(r"(\.| )", "_", task_name).strip().lower()
    disambiguation_changes = calculate_relative_improvements(df1, df2)
    task_id = task_name.split(" ")[-1]

    disambiguation_changes.style.format(
        lambda v: (
            f"+\\textcolor{{DarkGreen}}{{{v:.2f}}}"
            if v > 0
            else f"\\textcolor{{DarkRed}}{{{v:.2f}}}"
            if isinstance(v, float)
            else v
        ),
        na_rep="-",
    ).to_latex(
        report_dir / f"{task_name_formatted}_disambiguation.tex",
        caption=f"{mode} Set Disambiguation Changes for {task_name} between {task_id_to_name(f'{task_id}.1')} and {task_id_to_name(f'{task_id}.2')}.",
        label=f"tab:task:{task_name_formatted}{lbl_xtra}_disambiguation",
        clines="all;data",
        hrules=True,
    )
disambiguation_changes

$F_1$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter               
 Baseline -        -        -          -          -          -        -0.203689   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$ -0.039872   
                                                  \checkmark $\times$ -0.039872   
                            \checkmark $\times$   $\times$   $\times$ -0.029455   
                                                  \checkmark $\times$ -0.029455   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$ -0.022342   
                                                  \checkmark $\times$ -0.022342   
                            \checkmark $\times$   $\times$   $\times$ -0.018422   
                                                  \checkmark $\times$ -0.018422   
                                       \checkmark $\times$   $\times$ -0.011716   

                                                                       $F_{1,micro}$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter                    
 Baseline -        -        -          -          -          -             -0.254076   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$      -0.049461   
                                                  \checkmark $\times$      -0.049461   
                            \checkmark $\times$   $\times$   $\times$      -0.032104   
                                                  \checkmark $\times$      -0.032104   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$      -0.049691   
                                                  \checkmark $\times$      -0.049691   
                            \checkmark $\times$   $\times$   $\times$      -0.021077   
                                                  \checkmark $\times$      -0.021077   
                                       \checkmark $\times$   $\times$      -0.052747   

                                                                       Relative $F_1$  \
Model     Beams    NED FT   RAG        LoRA       Naive      Filter                     
 Baseline -        -        -          -          -          -              -0.678287   
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$       -0.614039   
                                                  \checkmark $\times$       -0.614039   
                            \checkmark $\times$   $\times$   $\times$       -0.463968   
                                                  \checkmark $\times$       -0.463968   
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$       -0.456127   
                                                  \checkmark $\times$       -0.456127   
                            \checkmark $\times$   $\times$   $\times$       -0.402148   
                                                  \checkmark $\times$       -0.402148   
                                       \checkmark $\times$   $\times$       -0.518017   

                                                                       Relative $F_{1,micro}$  
Model     Beams    NED FT   RAG        LoRA       Naive      Filter                            
 Baseline -        -        -          -          -          -                      -0.653809  
3.1 8B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$               -0.434143  
                                                  \checkmark $\times$               -0.434143  
                            \checkmark $\times$   $\times$   $\times$               -0.424432  
                                                  \checkmark $\times$               -0.424432  
3.2 3B    $\times$ $\times$ $\times$   \checkmark $\times$   $\times$               -0.436998  
                                                  \checkmark $\times$               -0.436998  
                            \checkmark $\times$   $\times$   $\times$               -0.